# Module 23 — Gradient Accumulation

Sometimes the batch size you want doesn't fit in GPU memory. **Gradient
accumulation** simulates a bigger batch by running several smaller
"micro-batches" — forward, backward, but *no* optimizer step — and letting
their gradients add up in `.grad` (which is exactly how `.backward()`
already behaves; Module 04 covered this). Only after `N` micro-batches does
`optimizer.step()` actually run. Divide the loss by `N` first, and the
result is mathematically **identical** to having run one big batch — this
module proves that directly, not just asserts it.

## 1. Proving exact equivalence to a single large batch

Same model, same data, same starting weights — one run processes all 16
examples at once; the other processes 4 micro-batches of 4 and
accumulates. The resulting gradients must match to floating-point
precision.

In [ ]:
import copy

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
model = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 10))
model_accum = copy.deepcopy(model)  # identical starting weights

x = torch.randn(16, 32)
y = torch.randint(0, 10, (16,))

# one big batch
model.zero_grad()
F.cross_entropy(model(x), y).backward()
full_batch_grads = [p.grad.clone() for p in model.parameters()]

# 4 micro-batches of 4, accumulated
accum_steps = 4
micro_batch_size = 16 // accum_steps
model_accum.zero_grad()
for i in range(accum_steps):
    xb = x[i * micro_batch_size:(i + 1) * micro_batch_size]
    yb = y[i * micro_batch_size:(i + 1) * micro_batch_size]
    loss = F.cross_entropy(model_accum(xb), yb) / accum_steps  # normalize by accum_steps
    loss.backward()
accumulated_grads = [p.grad.clone() for p in model_accum.parameters()]

for g_full, g_accum in zip(full_batch_grads, accumulated_grads):
    assert torch.allclose(g_full, g_accum, atol=1e-6), (g_full - g_accum).abs().max().item()
print("Confirmed: 4 accumulated micro-batches of 4 produce gradients identical to one batch of 16.")
print("(Dividing the loss by accum_steps is what makes the accumulated mean match the full-batch mean exactly.)")

## 2. The real payoff: lower peak memory, measured on this GPU

A bigger model, so the effect is large enough to see clearly. Same total
amount of data processed (`full_batch` examples) either as one batch or as
several accumulated micro-batches — only *how* it's split changes.

In [ ]:
assert torch.cuda.is_available()
device = "cuda"
torch.manual_seed(0)

big_model = nn.Sequential(*[nn.Linear(2048, 2048) for _ in range(6)], nn.Linear(2048, 10)).to(device)

full_batch = 4096
micro_batch = 512
accum_steps = full_batch // micro_batch

x = torch.randn(full_batch, 2048, device=device)
y = torch.randint(0, 10, (full_batch,), device=device)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
big_model.zero_grad()
F.cross_entropy(big_model(x), y).backward()
full_batch_peak = torch.cuda.max_memory_allocated() / 1e6

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
big_model.zero_grad()
for i in range(accum_steps):
    xb = x[i * micro_batch:(i + 1) * micro_batch]
    yb = y[i * micro_batch:(i + 1) * micro_batch]
    (F.cross_entropy(big_model(xb), yb) / accum_steps).backward()
accumulated_peak = torch.cuda.max_memory_allocated() / 1e6

print(f"One batch of {full_batch}:                    {full_batch_peak:.1f} MB peak")
print(f"{accum_steps} accumulated micro-batches of {micro_batch}: {accumulated_peak:.1f} MB peak")
print(f"Memory reduction: {(1 - accumulated_peak / full_batch_peak):.1%}")
assert accumulated_peak < full_batch_peak

## 3. Effective batch size

The number that matters for training dynamics (learning rate schedules,
Module 25) is the **effective** batch size — how many examples' worth of
gradient signal go into each optimizer step, regardless of how it's split
into micro-batches.

In [ ]:
per_device_batch_size = 512
gradient_accumulation_steps = 8
effective_batch_size = per_device_batch_size * gradient_accumulation_steps
print(f"effective batch size = {per_device_batch_size} x {gradient_accumulation_steps} = {effective_batch_size}")
print("This exact pattern - a per-device batch size plus an accumulation step count - is what Module 31\'s real pretraining run will configure.")

## Recap

- Gradient accumulation is exact, not an approximation — proven directly:
  accumulating over micro-batches (with the loss divided by the number of
  accumulation steps) gives bit-for-bit the same gradient as one large
  batch.
- Measured on this machine's GPU: splitting one batch into smaller
  accumulated micro-batches meaningfully reduced peak memory, letting a
  larger *effective* batch size fit than would otherwise be possible.
- Effective batch size = micro-batch size x accumulation steps — the
  number that actually matters for training dynamics.

Module 24 covers the AdamW optimizer itself — why it's the standard choice
for transformer training over plain SGD.